### Instruction

> 1. Rename assignment-01-XXX-YYY.ipynb where XXX is your student ID and YYY is your Chinese name.
> 2. The deadline of Assignment-01 is 23:59pm, 04-02-2026
> 3. In this assignment, you will
>    1) Explore Wikipedia text data
>    2) Build language models
>    3) Build NB and LR classifiers
>    4) Build embedding model for document classification
> 4. Submit your homework as html file just like our exercise did.
> Download the preprocessed data, enwiki-train.json and enwiki-test.json from the Assignment-01 folder. In the data file, each line contains a Wikipedia page with attributes, title, label, and text. There are 9000 records in the train file and 1000 records in test file with ten categories.

- Please print out all your outputs in the notebook and save it.

### Task1 - Data exploring

> 1) Print out how many documents are in each class  (for both train and test dataset)

In [1]:
# =========================
# Task 1.1 统计每个类别的文档数量
# =========================
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
import json
import math
import random
import re
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import spacy

# 设定随机种子，保证实验可复现
random.seed(42)
np.random.seed(42)

# 数据路径
TRAIN_PATH = Path(".datasets/assignment1/enwiki-train.json")
TEST_PATH = Path(".datasets/assignment1/enwiki-test.json")

# 读取函数 JSON Lines（每行一个 json 对象）
def load_wiki_data(path):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    # 按 json lines 逐行读取
    data = [json.loads(line) for line in text.splitlines() if line.strip()]

    df = pd.DataFrame(data)

    # 只保留本次作业需要的列，并检查是否存在
    needed_cols = ["title", "label", "text"]
    for col in needed_cols:
        if col not in df.columns:
            raise ValueError(f"数据中缺少必要字段：{col}")

    return df[needed_cols].copy()

# 读取训练集和测试集
df_train = load_wiki_data(TRAIN_PATH)
df_test = load_wiki_data(TEST_PATH)

print("训练集大小：", df_train.shape)
print("测试集大小：", df_test.shape)
print()

# 统一类别顺序，方便后续所有单元格保持一致
label_order = sorted(set(df_train["label"].tolist()) | set(df_test["label"].tolist()))

# 统计每个类别的文档数
train_counts = df_train["label"].value_counts().reindex(label_order, fill_value=0)
test_counts = df_test["label"].value_counts().reindex(label_order, fill_value=0)

print("=== 训练集每个类别的文档数量 ===")
print(train_counts.to_string())
print()

print("=== 测试集每个类别的文档数量 ===")
print(test_counts.to_string())

训练集大小： (9000, 3)
测试集大小： (1000, 3)

=== 训练集每个类别的文档数量 ===
label
Actor           79
Animal          82
Artist         457
Book           858
Disease        202
Film          2752
Food           121
Politician    3441
Software       239
Writer         769

=== 测试集每个类别的文档数量 ===
label
Actor           1
Animal         11
Artist         63
Book          117
Disease        18
Film          296
Food           16
Politician    383
Software       27
Writer         68


> 2) Print out the average number of sentences in each class.
>    You may need to use sentence tokenization tools from spacy for both train and test dataset.



In [2]:
# =========================
# Task 1.2 统计每个类别的平均句子数（加速版）
# =========================

# 使用 spaCy 的 sentencizer 做句子切分
# 这里不用下载大型英文模型，直接用轻量规则切句即可
nlp = spacy.blank("en")
if "sentencizer" not in nlp.pipe_names:
    nlp.add_pipe("sentencizer")

# 批量统计句子数，比对每一行单独调用 nlp(text) 更快
def count_sentences_batch(text_list, batch_size=512):
    counts = []
    
    # 把非字符串数据统一转为空字符串，避免报错
    safe_texts = [text if isinstance(text, str) else "" for text in text_list]
    
    # 使用 nlp.pipe 批量处理文本，加快速度
    for doc in nlp.pipe(safe_texts, batch_size=batch_size, n_process=16):
        # 只统计非空句子
        cnt = sum(1 for sent in doc.sents if sent.text.strip())
        counts.append(cnt)
    
    return counts

# 批量计算每篇文档的句子数
df_train["num_sentences"] = count_sentences_batch(df_train["text"].tolist(), batch_size=512)
df_test["num_sentences"] = count_sentences_batch(df_test["text"].tolist(), batch_size=512)

# 分类别求平均句子数
train_avg_sent = df_train.groupby("label")["num_sentences"].mean().reindex(label_order)
test_avg_sent = df_test.groupby("label")["num_sentences"].mean().reindex(label_order)

print("=== 训练集每个类别的平均句子数 ===")
for label, value in train_avg_sent.items():
    print(f"{label}: {value:.2f}")

print()
print("=== 测试集每个类别的平均句子数 ===")
for label, value in test_avg_sent.items():
    print(f"{label}: {value:.2f}")

=== 训练集每个类别的平均句子数 ===
Actor: 69.33
Animal: 67.06
Artist: 185.94
Book: 205.08
Disease: 347.86
Film: 177.88
Food: 153.44
Politician: 222.86
Software: 201.13
Writer: 216.11

=== 测试集每个类别的平均句子数 ===
Actor: 177.00
Animal: 62.82
Artist: 174.97
Book: 197.96
Disease: 325.89
Film: 173.68
Food: 165.12
Politician: 233.03
Software: 204.00
Writer: 220.60


> 3) Print out the average number of tokens in each class for both train and test dataset.

In [3]:
# =========================
# Task 1.3 统计每个类别的平均 token 数
# =========================

# 这里把 token 定义为：只保留英文单词和数字
# 例如 hello、abc123、2024 都算 token
def simple_word_tokenize(text):
    text = text.lower()
    tokens = re.findall(r"[a-z0-9]+", text)
    return tokens

# 计算每篇文档的 token 数
df_train["num_tokens"] = df_train["text"].apply(lambda x: len(simple_word_tokenize(x)))
df_test["num_tokens"] = df_test["text"].apply(lambda x: len(simple_word_tokenize(x)))

# 分类别求平均 token 数
train_avg_tok = df_train.groupby("label")["num_tokens"].mean().reindex(label_order)
test_avg_tok = df_test.groupby("label")["num_tokens"].mean().reindex(label_order)

print("=== 训练集每个类别的平均 token 数 ===")
for label, value in train_avg_tok.items():
    print(f"{label}: {value:.2f}")

print()
print("=== 测试集每个类别的平均 token 数 ===")
for label, value in test_avg_tok.items():
    print(f"{label}: {value:.2f}")

=== 训练集每个类别的平均 token 数 ===
Actor: 1407.67
Animal: 1237.35
Artist: 4178.81
Book: 4617.99
Disease: 7137.75
Film: 3840.17
Food: 2941.41
Politician: 5063.04
Software: 4278.90
Writer: 4927.83

=== 测试集每个类别的平均 token 数 ===
Actor: 3627.00
Animal: 1125.55
Artist: 3940.14
Book: 4456.99
Disease: 6984.33
Film: 3758.93
Food: 2992.69
Politician: 5290.55
Software: 4215.67
Writer: 5047.65


> 4) For each sentence in the document, remove punctuations and other special characters so that each sentence only contains English words and numbers. To make your life easier, you can make all words as lower cases. For each class, print out the first article's name and the processed first 40 words. (for both train and test dataset)

In [4]:
# =========================
# Task 1.4 文本清洗，并输出每个类别第一篇文章的标题和前 40 个词（加速版）
# =========================

import re

# 预编译正则表达式，避免重复编译，提高速度
token_pattern = re.compile(r"[a-z0-9]+")

# 把一句文本清洗成 token 列表
def normalize_tokens(text):
    # 转小写后，只保留英文和数字
    return token_pattern.findall(text.lower())

# 批量清洗文本：
# 一次性生成 clean_sentences / clean_tokens / clean_text
def batch_clean_texts(text_list, batch_size=512):
    clean_sentences_all = []
    clean_tokens_all = []
    clean_text_all = []

    # 把非字符串数据统一转为空字符串，避免报错
    safe_texts = [text if isinstance(text, str) else "" for text in text_list]

    # 使用 nlp.pipe 批量切句，比逐条 nlp(text) 更快
    for doc in nlp.pipe(safe_texts, batch_size=batch_size, n_process=16):
        sent_tokens_list = []
        flat_tokens = []

        # 遍历当前文档的每个句子
        for sent in doc.sents:
            toks = normalize_tokens(sent.text)
            if toks:
                sent_tokens_list.append(toks)
                flat_tokens.extend(toks)

        clean_sentences_all.append(sent_tokens_list)
        clean_tokens_all.append(flat_tokens)
        clean_text_all.append(" ".join(flat_tokens))

    return clean_sentences_all, clean_tokens_all, clean_text_all

# 对训练集和测试集分别批量处理
for df in [df_train, df_test]:
    clean_sentences, clean_tokens, clean_text = batch_clean_texts(
        df["text"].tolist(),
        batch_size=256
    )

    df["clean_sentences"] = clean_sentences
    df["clean_tokens"] = clean_tokens
    df["clean_text"] = clean_text

# 打印每个类别中“第一篇文章”的标题和处理后的前 40 个词
def print_first_article_preview(df, split_name):
    print(f"===== {split_name} =====")
    for label in label_order:
        sub_df = df[df["label"] == label]
        if len(sub_df) == 0:
            continue

        # 取该类别第一篇文章（保持原数据顺序）
        row = sub_df.iloc[0]
        first_40_words = row["clean_tokens"][:40]

        print(f"\n类别: {label}")
        print(f"标题: {row['title']}")
        print("前40个处理后的词:")
        print(" ".join(first_40_words))

print_first_article_preview(df_train, "训练集")
print()
print_first_article_preview(df_test, "测试集")

===== 训练集 =====

类别: Actor
标题: Laura_Bonarrigo
前40个处理后的词:
laura bonarrigo koffman born laura bonarrigo october 29 1964 is an american actress early life laura bonarrigo was born in brookline massachusetts she became a member of the shoestring players a professional children s theater group while still in grade

类别: Animal
标题: Lunaspis
前40个处理后的词:
lunaspis is an extinct genus of armor plated petalichthyid placoderm fish that lived in shallow marine environments of the early devonian period from approximately 409 1 to 402 5 million year ago fossils have been found in germany china and

类别: Artist
标题: Dan_Graham
前40个处理后的词:
daniel graham born march 31 1942 is an american artist writer and curator graham grew up in new jersey in 1964 he began directing the john daniels gallery in new york where he put on sol lewitt s first one

类别: Book
标题: Middlesex_(novel)
前40个处理后的词:
middlesex is a pulitzer prize winning novel by jeffrey eugenides published in 2002 the book is a bestseller with more than f

### Task2 - Build language models

Before you go, you should do necessary preprocessing for training and testing text. For example, you should do sentence tokenization, removing special characters, replacing less frequency words as UNK (for example, you can try to use a cutoff of 10), making all words as lower characters. Fix your vocabulary size so that is not too large.

> 1) Based on the training dataset (collect all sentences in training dataset), build unigram, bigram, and trigram language models using Additive smoothing technique. It is encouraged to implement models by yourself.


In [5]:
# =========================
# Task 2.1 自己实现 unigram / bigram / trigram 语言模型（Additive smoothing）
# =========================

# 先收集训练集与测试集中的“句子级 token 序列”
# 语言模型要在句子级别训练，而不是整篇文档直接拼起来
train_sentence_tokens_raw = []
for sent_list in df_train["clean_sentences"]:
    train_sentence_tokens_raw.extend(sent_list)

test_sentence_tokens_raw = []
for sent_list in df_test["clean_sentences"]:
    test_sentence_tokens_raw.extend(sent_list)

print("训练集句子总数：", len(train_sentence_tokens_raw))
print("测试集句子总数：", len(test_sentence_tokens_raw))

# 统计训练集中的词频
word_freq = Counter(tok for sent in train_sentence_tokens_raw for tok in sent)

# 设置低频词截断阈值
CUTOFF = 10

# 只保留频率 >= CUTOFF 的词进入词表
vocab_words = {w for w, c in word_freq.items() if c >= CUTOFF}

# 语言模型的特殊符号
UNK = "<UNK>"
BOS = "<BOS>"
EOS = "<EOS>"

lm_vocab = set(vocab_words) | {UNK, BOS, EOS}

print("原始训练集不同词数：", len(word_freq))
print("截断后词表大小：", len(lm_vocab))

# 用训练集词表替换低频词/OOV 为 <UNK>
def replace_rare_with_unk(sentence_tokens, vocab_words):
    return [tok if tok in vocab_words else UNK for tok in sentence_tokens]

train_sentences_lm = [replace_rare_with_unk(sent, vocab_words) for sent in train_sentence_tokens_raw if len(sent) > 0]
test_sentences_lm = [replace_rare_with_unk(sent, vocab_words) for sent in test_sentence_tokens_raw if len(sent) > 0]

# 定义一个通用 N-gram 语言模型类
class NGramLM:
    def __init__(self, n=1, alpha=1.0, vocab=None):
        self.n = n                    # n 的取值：1/2/3
        self.alpha = alpha            # 加性平滑参数
        self.vocab = sorted(vocab) if vocab is not None else []
        self.vocab_set = set(self.vocab)
        self.V = len(self.vocab)      # 词表大小
        
        # ngram_counts: 统计 n-gram 次数
        # context_counts: 统计上下文次数
        self.ngram_counts = Counter()
        self.context_counts = Counter()
        self.total_tokens = 0         # 仅 unigram 会直接用到

    # 给句子加上 BOS / EOS
    def _prepare_sentence(self, sentence_tokens):
        mapped = [tok if tok in self.vocab_set else UNK for tok in sentence_tokens]
        if self.n == 1:
            return mapped + [EOS]
        else:
            return [BOS] * (self.n - 1) + mapped + [EOS]

    # 训练语言模型
    def fit(self, sentences):
        for sent in sentences:
            tokens = self._prepare_sentence(sent)

            # unigram 单独处理
            if self.n == 1:
                for tok in tokens:
                    self.ngram_counts[(tok,)] += 1
                    self.total_tokens += 1
                continue

            # bigram / trigram
            for i in range(self.n - 1, len(tokens)):
                ngram = tuple(tokens[i - self.n + 1 : i + 1])
                context = ngram[:-1]
                self.ngram_counts[ngram] += 1
                self.context_counts[context] += 1

        if self.n == 1:
            self.context_counts[()] = self.total_tokens

        return self

    # 计算 P(token | context)
    def prob(self, token, context=None):
        token = token if token in self.vocab_set else UNK

        if self.n == 1:
            count = self.ngram_counts[(token,)]
            return (count + self.alpha) / (self.total_tokens + self.alpha * self.V)

        context = context or []
        context = [tok if tok in self.vocab_set else UNK for tok in context]
        context = tuple(context[-(self.n - 1):])

        count = self.ngram_counts[context + (token,)]
        context_count = self.context_counts[context]

        return (count + self.alpha) / (context_count + self.alpha * self.V)

    # 计算一句话的对数概率
    def sentence_log_prob(self, sentence_tokens):
        tokens = self._prepare_sentence(sentence_tokens)
        log_prob = 0.0

        if self.n == 1:
            for tok in tokens:
                log_prob += math.log(self.prob(tok))
            return log_prob

        for i in range(self.n - 1, len(tokens)):
            context = tokens[i - self.n + 1 : i]
            token = tokens[i]
            log_prob += math.log(self.prob(token, context))

        return log_prob

    # 计算整个语料的困惑度
    def perplexity(self, sentences):
        total_log_prob = 0.0
        total_pred_tokens = 0

        for sent in sentences:
            total_log_prob += self.sentence_log_prob(sent)
            # 一般把 EOS 也算进预测 token 数
            total_pred_tokens += len(sent) + 1

        return math.exp(-total_log_prob / total_pred_tokens)

    # 生成一句话
    def generate_sentence(self, max_len=20, seed=None):
        if seed is not None:
            random.seed(seed)

        # 初始上下文
        context = [BOS] * (self.n - 1) if self.n > 1 else []
        generated = []

        # 生成时不让 BOS 出现在候选里
        candidates = [tok for tok in self.vocab if tok != BOS]

        for _ in range(max_len):
            probs = [self.prob(tok, context) for tok in candidates]
            next_tok = random.choices(candidates, weights=probs, k=1)[0]

            if next_tok == EOS:
                break

            # 为了让输出更好看，生成阶段跳过 UNK
            if next_tok != UNK:
                generated.append(next_tok)

            if self.n > 1:
                context = (context + [next_tok])[-(self.n - 1):]

        return " ".join(generated) if len(generated) > 0 else "(空句子)"

# 训练三种语言模型
unigram_lm = NGramLM(n=1, alpha=1.0, vocab=lm_vocab).fit(train_sentences_lm)
bigram_lm = NGramLM(n=2, alpha=1.0, vocab=lm_vocab).fit(train_sentences_lm)
trigram_lm = NGramLM(n=3, alpha=1.0, vocab=lm_vocab).fit(train_sentences_lm)

print("语言模型训练完成。")
print("Unigram / Bigram / Trigram 已建立。")

训练集句子总数： 1831229
测试集句子总数： 204705
原始训练集不同词数： 320197
截断后词表大小： 75535
语言模型训练完成。
Unigram / Bigram / Trigram 已建立。


> 2) Report the perplexity of these 3 trained models on the testing dataset (again collect all sentences in training dataset) and explain your findings. 

In [ ]:
# =========================
# Task 2.2 在测试集上计算三种语言模型的 perplexity
# =========================

unigram_ppl = unigram_lm.perplexity(test_sentences_lm)
bigram_ppl = bigram_lm.perplexity(test_sentences_lm)
trigram_ppl = trigram_lm.perplexity(test_sentences_lm)

print("=== 测试集 Perplexity ===")
print(f"Unigram Perplexity : {unigram_ppl:.4f}")
print(f"Bigram  Perplexity : {bigram_ppl:.4f}")
print(f"Trigram Perplexity : {trigram_ppl:.4f}")

# 一个简短解释
ppl_results = {
    "unigram": unigram_ppl,
    "bigram": bigram_ppl,
    "trigram": trigram_ppl
}
best_model = min(ppl_results, key=ppl_results.get)

print("\n=== 简要解释 ===")
print("1. Perplexity 越低，说明模型对测试集的预测越好。")
print("2. unigram 只看当前词，不看上下文；bigram / trigram 会利用更多上下文信息。")
print("3. 但 n 越大也越容易出现数据稀疏问题，因此需要平滑。")
print(f"4. 在本次实验中，困惑度最低的是：{best_model}。")

=== 测试集 Perplexity ===
Unigram Perplexity : 1466.5092
Bigram  Perplexity : 1524.7748
Trigram Perplexity : 11275.1770

=== 简要解释 ===
1. Perplexity 越低，说明模型对测试集的预测越好。
2. unigram 只看当前词，不看上下文；bigram / trigram 会利用更多上下文信息。
3. 但 n 越大也越容易出现数据稀疏问题，因此需要平滑。
4. 在本次实验中，困惑度最低的是：unigram。


> 3) Use each built model to generate five sentences and explain these generated patterns.


In [7]:
# =========================
# Task 2.3 用每个语言模型各生成 5 个句子
# =========================

random.seed(42)

print("=== Unigram 生成的 5 个句子 ===")
for i in range(5):
    print(f"{i+1}. {unigram_lm.generate_sentence(max_len=20)}")

print("\n=== Bigram 生成的 5 个句子 ===")
for i in range(5):
    print(f"{i+1}. {bigram_lm.generate_sentence(max_len=20)}")

print("\n=== Trigram 生成的 5 个句子 ===")
for i in range(5):
    print(f"{i+1}. {trigram_lm.generate_sentence(max_len=20)}")

print("\n=== 生成模式说明 ===")
print("1. Unigram 只根据单词整体频率采样，所以句子通常语法性较差、主题跳跃明显。")
print("2. Bigram 会考虑前一个词，因此局部搭配会比 unigram 更自然。")
print("3. Trigram 会考虑前两个词，通常在短距离语法和固定短语上表现更好。")
print("4. 但高阶模型也更容易受训练数据稀疏影响，可能生成较短或较保守的句子。")

=== Unigram 生成的 5 个句子 ===
1. orders 98 cosmic brought school primary to a him
2. both jump
3. bbc paris masilamani brandes od the 1917 that reform father and way faces a a the of the s made
4. who fur mercury the of the newspaper reported
5. but deceased by a court openly for france beloved conditions vestiges painter of army s anthy gain world orgy

=== Bigram 生成的 5 个句子 ===
1. loud pinpoint sparsely rpg curragh accentuated ewell distanced cooks usaid swathed evictions paf hager trauma intimated disinterred democr mauled discharge
2. modoc thievery hangover cpac zhongnanhai lara azur alarmed benamou northfield saxby honeycutt antennae gp zbor loadable wechat straps 1981 pushkin
3. rain lugard disrupt openbsd berghof iemma instructing vest swamped discounted krasner choirs transcend sunita emotion oiu nandini camarilla reznor lyndon
4. the lyrics 034 faint 566 unambiguously symons slr equestrian analyzing symptomatology valter auster katsav apricot revathi rindge bonifacio jojo maneuver

> 4) Train bigram and trigram model using kenlm and report the perplexities of these two. Compare results of your model and results from kenlm

In [ ]:
# =========================
# Task 2.4 使用 kenlm 训练 bigram / trigram，并比较 perplexity
# =========================

import subprocess
from pathlib import Path
import kenlm

kenlm_dir = Path("kenlm_work")
kenlm_dir.mkdir(exist_ok=True)

train_txt_path = kenlm_dir / "train_sentences.txt"
with open(train_txt_path, "w", encoding="utf-8") as f:
    for sent in train_sentences_lm:
        f.write(" ".join(sent) + "\n")

print("已写出训练语料：", train_txt_path)

lmplz_path = "/data/yc/llm-26-homework/kenlm/build/bin/lmplz"

bigram_arpa = kenlm_dir / "bigram.arpa"
trigram_arpa = kenlm_dir / "trigram.arpa"

cmd_bigram = f'"{lmplz_path}" -o 2 --discount_fallback < "{train_txt_path}" > "{bigram_arpa}"'
cmd_trigram = f'"{lmplz_path}" -o 3 --discount_fallback < "{train_txt_path}" > "{trigram_arpa}"'

subprocess.run(cmd_bigram, shell=True, check=True)
subprocess.run(cmd_trigram, shell=True, check=True)

print("KenLM 模型训练完成：")
print(" -", bigram_arpa)
print(" -", trigram_arpa)

kenlm_bigram = kenlm.Model(str(bigram_arpa))
kenlm_trigram = kenlm.Model(str(trigram_arpa))

def kenlm_perplexity(model, tokenized_sentences):
    total_log10_prob = 0.0
    total_tokens = 0

    for sent in tokenized_sentences:
        sent_str = " ".join(sent)
        total_log10_prob += model.score(sent_str, bos=True, eos=True)
        total_tokens += len(sent) + 1

    return 10 ** (-total_log10_prob / total_tokens)

kenlm_bigram_ppl = kenlm_perplexity(kenlm_bigram, test_sentences_lm)
kenlm_trigram_ppl = kenlm_perplexity(kenlm_trigram, test_sentences_lm)

print("\n=== 自己实现 vs KenLM 的 Perplexity 对比 ===")
print(f"自己实现 Bigram  PPL : {bigram_ppl:.4f}")
print(f"KenLM    Bigram  PPL : {kenlm_bigram_ppl:.4f}")
print(f"自己实现 Trigram PPL : {trigram_ppl:.4f}")
print(f"KenLM    Trigram PPL : {kenlm_trigram_ppl:.4f}")

已写出训练语料： kenlm_work/train_sentences.txt


=== 1/5 Counting and sorting n-grams ===
Reading /data/yc/llm-26-homework/kenlm_work/train_sentences.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 40684581 types 75536
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:906432 2:432374448128
Statistics:
1 75536 D1=0.331617 D2=0.874885 D3+=1.16764
2 6460653 D1=0.716792 D2=1.0787 D3+=1.38539
Memory estimate for binary LM:
type     MB
probing 112 assuming -p 1.5
probing 113 assuming -r models -p 1.5
trie     38 without quantization
trie     20 assuming -q 8 -b 8 quantization 
trie     38 assuming -a 22 array pointer compression
trie     20 assuming -a 22 -q 8 -b 8 array pointer compression and quantization
=== 3/5 Calculating and sorting initial probabilities ===
Chain sizes: 1:906432 2:103370448
=== 4/5 Calculating and writing order-interpolated proba

KenLM 模型训练完成：
 - kenlm_work/bigram.arpa
 - kenlm_work/trigram.arpa


Loading the LM will be faster if you build a binary file.
Reading /data/yc/llm-26-homework/kenlm_work/bigram.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Loading the LM will be faster if you build a binary file.
Reading /data/yc/llm-26-homework/kenlm_work/trigram.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************



=== 自己实现 vs KenLM 的 Perplexity 对比 ===
自己实现 Bigram  PPL : 1524.7748
KenLM    Bigram  PPL : 353.4368
自己实现 Trigram PPL : 11275.1770
KenLM    Trigram PPL : 248.4485


## Task3 - Build NB/LR classifiers

> 1) Build a Naive Bayes classifier (with Laplace smoothing) and test your model on test dataset

In [9]:
# =========================
# Task 3.1 自己实现 Naive Bayes（Laplace smoothing）
# =========================

# 这里实现多项式朴素贝叶斯（Multinomial NB）
# 输入特征使用清洗后的单词序列 clean_tokens

class MultinomialNBFromScratch:
    def __init__(self, alpha=1.0):
        self.alpha = alpha  # 拉普拉斯平滑系数

    def fit(self, X_tokens, y):
        # 所有类别
        self.classes_ = sorted(set(y))

        # 训练集词表
        self.vocab_ = {tok for doc in X_tokens for tok in doc}
        self.vocab_.add("<UNK>")
        self.V_ = len(self.vocab_)

        # 先验概率 P(class)
        self.class_doc_count_ = Counter(y)
        total_docs = len(y)
        self.log_prior_ = {
            c: math.log(self.class_doc_count_[c] / total_docs)
            for c in self.classes_
        }

        # 条件概率 P(word | class) 所需计数
        self.word_count_ = {c: Counter() for c in self.classes_}
        self.total_word_count_ = {c: 0 for c in self.classes_}

        for tokens, label in zip(X_tokens, y):
            mapped = [tok if tok in self.vocab_ else "<UNK>" for tok in tokens]
            self.word_count_[label].update(mapped)
            self.total_word_count_[label] += len(mapped)

        return self

    def predict_one(self, tokens):
        mapped = [tok if tok in self.vocab_ else "<UNK>" for tok in tokens]

        scores = {}
        for c in self.classes_:
            # 从 log P(class) 开始
            score = self.log_prior_[c]

            # 分母：该类下所有词总数 + alpha * V
            denom = self.total_word_count_[c] + self.alpha * self.V_

            # 累加 log P(word | class)
            for tok in mapped:
                count = self.word_count_[c][tok]
                score += math.log((count + self.alpha) / denom)

            scores[c] = score

        # 取后验概率最大的类别
        return max(scores, key=scores.get)

    def predict(self, X_tokens):
        return [self.predict_one(tokens) for tokens in X_tokens]

# 训练 NB 模型
nb_model = MultinomialNBFromScratch(alpha=1.0)
nb_model.fit(df_train["clean_tokens"].tolist(), df_train["label"].tolist())

# 在测试集上预测
nb_pred = nb_model.predict(df_test["clean_tokens"].tolist())

# 输出简单结果
print("Naive Bayes 预测完成。")
print("测试集前 20 条预测结果：")
print(nb_pred[:20])

Naive Bayes 预测完成。
测试集前 20 条预测结果：
['Politician', 'Book', 'Book', 'Film', 'Film', 'Book', 'Writer', 'Politician', 'Politician', 'Artist', 'Book', 'Film', 'Film', 'Film', 'Artist', 'Writer', 'Politician', 'Politician', 'Film', 'Food']


> 2) Build a LR classifier. This question seems to be challenging. We did not directly provide features for samples. But just use your own method to build useful features. You may need to split the training dataset into train and validation so that some involved parameters can be tuned. 

In [10]:
# =========================
# Task 3.2 构建 LR 分类器
# 这里采用：TF-IDF(1-2gram) + Logistic Regression
# =========================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# 准备文本与标签
X_all = df_train["clean_text"].tolist()
y_all = df_train["label"].tolist()
X_test_lr = df_test["clean_text"].tolist()

# 先把训练集切分成 train / validation，用于调参
X_tr, X_val, y_tr, y_val = train_test_split(
    X_all, y_all,
    test_size=0.15,
    random_state=42,
    stratify=y_all
)

# 建立 TF-IDF 向量器
# 因为文本已经预处理过，所以 tokenizer 直接用 str.split
vectorizer = TfidfVectorizer(
    lowercase=False,
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    ngram_range=(1, 2),     # 用 unigram + bigram 特征
    min_df=3,               # 过滤过低频的特征
    max_df=0.9,             # 过滤过高频的特征
    max_features=30000,
    sublinear_tf=True
)

# 在训练子集上拟合向量器
X_tr_vec = vectorizer.fit_transform(X_tr)
X_val_vec = vectorizer.transform(X_val)

# 调参：尝试不同的 C，选择验证集 Macro-F1 最好的模型
candidate_C = [0.1, 0.5, 1.0, 2.0, 5.0]
best_c = None
best_val_macro_f1 = -1
best_lr_model = None

for c in candidate_C:
    lr = LogisticRegression(
        C=c,
        max_iter=2000,
        solver="lbfgs",
    )
    lr.fit(X_tr_vec, y_tr)
    val_pred = lr.predict(X_val_vec)
    val_macro_f1 = f1_score(y_val, val_pred, average="macro")

    print(f"C = {c:<4} -> Validation Macro-F1 = {val_macro_f1:.4f}")

    if val_macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_macro_f1
        best_c = c
        best_lr_model = lr

print("\n最优 C =", best_c)
print("最佳验证集 Macro-F1 =", round(best_val_macro_f1, 4))

# 用“全部训练集”重新训练最终 LR 模型
final_vectorizer = TfidfVectorizer(
    lowercase=False,
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.9,
    max_features=30000,
    sublinear_tf=True
)

X_train_final_vec = final_vectorizer.fit_transform(X_all)
X_test_final_vec = final_vectorizer.transform(X_test_lr)

lr_model = LogisticRegression(
    C=best_c,
    max_iter=2000,
    solver="lbfgs",
)
lr_model.fit(X_train_final_vec, y_all)

# 在测试集上预测
lr_pred = lr_model.predict(X_test_final_vec)

print("\nLogistic Regression 预测完成。")
print("测试集前 20 条预测结果：")
print(lr_pred[:20])

C = 0.1  -> Validation Macro-F1 = 0.6664
C = 0.5  -> Validation Macro-F1 = 0.8639
C = 1.0  -> Validation Macro-F1 = 0.9209
C = 2.0  -> Validation Macro-F1 = 0.9349
C = 5.0  -> Validation Macro-F1 = 0.9406

最优 C = 5.0
最佳验证集 Macro-F1 = 0.9406

Logistic Regression 预测完成。
测试集前 20 条预测结果：
['Politician' 'Book' 'Book' 'Film' 'Film' 'Book' 'Book' 'Politician'
 'Politician' 'Artist' 'Book' 'Film' 'Actor' 'Film' 'Artist' 'Politician'
 'Politician' 'Politician' 'Film' 'Food']


> 3) Report Micro-F1 score and Macro-F1 score for these classifiers on testing dataset explain your results.

In [11]:
# =========================
# Task 3.3 计算 NB / LR 的 Micro-F1 与 Macro-F1，并解释结果
# =========================

from sklearn.metrics import f1_score, classification_report

y_test = df_test["label"].tolist()

# NB 的指标
nb_micro_f1 = f1_score(y_test, nb_pred, average="micro")
nb_macro_f1 = f1_score(y_test, nb_pred, average="macro")

# LR 的指标
lr_micro_f1 = f1_score(y_test, lr_pred, average="micro")
lr_macro_f1 = f1_score(y_test, lr_pred, average="macro")

print("=== Naive Bayes ===")
print(f"Micro-F1: {nb_micro_f1:.4f}")
print(f"Macro-F1: {nb_macro_f1:.4f}")
print("\n分类报告：")
print(classification_report(y_test, nb_pred, digits=4))

print("\n=== Logistic Regression ===")
print(f"Micro-F1: {lr_micro_f1:.4f}")
print(f"Macro-F1: {lr_macro_f1:.4f}")
print("\n分类报告：")
print(classification_report(y_test, lr_pred, digits=4))

print("\n=== 结果解释 ===")
print("1. Micro-F1 会把所有样本放在一起统计，因此更受大类别影响。")
print("2. Macro-F1 会先算每个类别的 F1，再做平均，因此更关注各类别是否均衡表现。")

if lr_macro_f1 >= nb_macro_f1:
    print("3. 本实验中 LR 的 Macro-F1 不低于 NB，说明 TF-IDF + LR 更能利用区分性特征。")
    print("4. 朴素贝叶斯假设词之间条件独立，这个假设通常较强，因此效果可能不如 LR。")
else:
    print("3. 本实验中 NB 的 Macro-F1 更高，说明在当前预处理与特征设置下，NB 更适合这批数据。")
    print("4. 这也可能与 LR 的特征选择、正则化参数或文本长度分布有关。")

=== Naive Bayes ===
Micro-F1: 0.9340
Macro-F1: 0.7260

分类报告：
              precision    recall  f1-score   support

       Actor     0.0000    0.0000    0.0000         1
      Animal     0.0000    0.0000    0.0000        11
      Artist     0.9828    0.9048    0.9421        63
        Book     0.8839    0.8462    0.8646       117
     Disease     0.6071    0.9444    0.7391        18
        Film     0.9667    0.9797    0.9732       296
        Food     1.0000    1.0000    1.0000        16
  Politician     0.9688    0.9713    0.9700       383
    Software     0.9643    1.0000    0.9818        27
      Writer     0.7568    0.8235    0.7887        68

    accuracy                         0.9340      1000
   macro avg     0.7130    0.7470    0.7260      1000
weighted avg     0.9269    0.9340    0.9295      1000


=== Logistic Regression ===
Micro-F1: 0.9780
Macro-F1: 0.9777

分类报告：
              precision    recall  f1-score   support

       Actor     1.0000    1.0000    1.0000         1
 

/data/yc/miniconda/envs/llm-26-gpu/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/data/yc/miniconda/envs/llm-26-gpu/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/data/yc/miniconda/envs/llm-26-gpu/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c

### Task 4 - Classify documents using embeddings

> For each document (both training and testing documents), you have several choices to generate a document embedding from the embedding we trained in Task 1 (**Just choose one of them**):

> - Use the average of known static embeddings of all words in each document. Or use the first paragraph’s words and take an average on these embeddings.
> - Use the doc2vec algorithm to present each document.
> - Use modern embedding like Qwen-embedding 0.6b ...

> Build a classifer on training documents and testing this classifer on testing documents. Since you have the ground truth, you can use the Micro/Macro F1-score to measure the performance of these choices on testing documents.

In [ ]:
# =========================
# Task 4 使用现代 embedding（Qwen3-Embedding-0.6B）生成文档向量，
# 再用 Logistic Regression 做文档分类
# AutoTokenizer + AutoModel + mean pooling
# =========================
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

# -------------------------
# 1. 加载 tokenizer 和 model
# -------------------------

MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    padding_side="left",
    cache_dir="hf_cache"
)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    cache_dir="hf_cache",
    attn_implementation="flash_attention_2",
    dtype=torch.float16
).cuda()

# 切换到推理模式
model.eval()

print("模型加载完成。")


# -------------------------
# 2. 文本预处理
# -------------------------
# 现代 embedding 模型建议尽量使用原始文本，而不是 clean_text
# 因为 clean_text 会丢掉句法和自然语言结构信息

def prepare_doc_text(text):
    # 统一处理空值
    if not isinstance(text, str):
        return ""
    
    text = text.strip()
    
    return text

X_train_text = df_train["text"].apply(prepare_doc_text).tolist()
X_test_text = df_test["text"].apply(prepare_doc_text).tolist()

y_train_emb = df_train["label"].tolist()
y_test_emb = df_test["label"].tolist()

print("训练集文档数：", len(X_train_text))
print("测试集文档数：", len(X_test_text))


# -------------------------
# 3. 定义 pooling 函数
# -------------------------
# AutoModel 输出的是每个 token 的 hidden states
# 这里使用 mean pooling，把一个文档压成一个固定长度向量

def mean_pooling(last_hidden_state, attention_mask):
    # last_hidden_state: [batch_size, seq_len, hidden_size]
    # attention_mask:    [batch_size, seq_len]
    
    # 先把 mask 扩展到 hidden_size 维度
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    
    # 对有效 token 的向量求和
    summed = torch.sum(last_hidden_state * mask, dim=1)
    
    # 统计每条样本有效 token 的数量，避免除以 0
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    
    # 求平均
    return summed / counts


# -------------------------
# 4. 批量编码函数
# -------------------------
# 这里把文本批量送进模型，得到文档 embedding

@torch.no_grad()
def encode_texts(texts, batch_size=4, max_length=8192):
    all_embeddings = []
    
    for start_idx in range(0, len(texts), batch_size):
        batch_texts = texts[start_idx:start_idx + batch_size]
        
        # 分词并转成张量
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,   # 按 token 截断
            return_tensors="pt"
        )
        
        # 搬到 GPU
        inputs = {k: v.cuda() for k, v in inputs.items()}
        
        # 前向计算
        outputs = model(**inputs)
        
        # 使用 last_hidden_state 做 mean pooling
        emb = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])
        
        # 做 L2 归一化，通常更稳定
        emb = F.normalize(emb, p=2, dim=1)
        
        # 搬回 CPU，转 numpy
        all_embeddings.append(emb.cpu().float().numpy())
        
        if (start_idx // batch_size + 1) % 20 == 0:
            print(f"已编码 {(start_idx + len(batch_texts))}/{len(texts)} 条文本")
    
    return np.vstack(all_embeddings)


# -------------------------
# 5. 生成训练集和测试集 embedding
# -------------------------

BATCH_SIZE = 8
MAX_LENGTH = 8192

print("\n开始编码训练集...")
X_train_emb = encode_texts(
    X_train_text,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

print("\n开始编码测试集...")
X_test_emb = encode_texts(
    X_test_text,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

print("\n训练集 embedding 形状：", X_train_emb.shape)
print("测试集 embedding 形状：", X_test_emb.shape)


# -------------------------
# 6. 用 Logistic Regression 做分类
# -------------------------

emb_clf = LogisticRegression(
    max_iter=3000,
    solver="lbfgs"
)

emb_clf.fit(X_train_emb, y_train_emb)
emb_pred = emb_clf.predict(X_test_emb)


# -------------------------
# 7. 评估结果
# -------------------------
emb_micro_f1 = f1_score(y_test_emb, emb_pred, average="micro")
emb_macro_f1 = f1_score(y_test_emb, emb_pred, average="macro")

print("\n=== Modern Embedding + LR 的测试集结果 ===")
print(f"Micro-F1: {emb_micro_f1:.4f}")
print(f"Macro-F1: {emb_macro_f1:.4f}")

print("\n分类报告：")
print(classification_report(y_test_emb, emb_pred, digits=4, zero_division=0))


# -------------------------
# 8. 输出简要说明
# -------------------------
print("\n=== 结果说明 ===")
print("1. 本实验使用现代预训练 embedding 模型将整篇文档编码为稠密向量。")
print("2. 与静态词向量平均相比，现代 embedding 通常能更好地保留上下文和语义信息。")
print("3. 然后使用 Logistic Regression 在训练集向量上训练分类器，并在测试集上评估。")
print("4. 如果结果优于 Word2Vec 平均向量，说明现代 embedding 的文档表示能力更强。")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

模型加载完成。
训练集文档数： 9000
测试集文档数： 1000

开始编码训练集...
已编码 160/9000 条文本
已编码 320/9000 条文本
已编码 480/9000 条文本
已编码 640/9000 条文本
已编码 800/9000 条文本
已编码 960/9000 条文本
已编码 1120/9000 条文本
已编码 1280/9000 条文本
已编码 1440/9000 条文本
已编码 1600/9000 条文本
已编码 1760/9000 条文本
已编码 1920/9000 条文本
已编码 2080/9000 条文本
已编码 2240/9000 条文本
已编码 2400/9000 条文本
已编码 2560/9000 条文本
已编码 2720/9000 条文本
已编码 2880/9000 条文本
已编码 3040/9000 条文本
已编码 3200/9000 条文本
已编码 3360/9000 条文本
已编码 3520/9000 条文本
已编码 3680/9000 条文本
已编码 3840/9000 条文本
已编码 4000/9000 条文本
已编码 4160/9000 条文本
已编码 4320/9000 条文本
已编码 4480/9000 条文本
已编码 4640/9000 条文本
已编码 4800/9000 条文本
已编码 4960/9000 条文本
已编码 5120/9000 条文本
已编码 5280/9000 条文本
已编码 5440/9000 条文本
已编码 5600/9000 条文本
已编码 5760/9000 条文本
已编码 5920/9000 条文本
已编码 6080/9000 条文本
已编码 6240/9000 条文本
已编码 6400/9000 条文本
已编码 6560/9000 条文本
已编码 6720/9000 条文本
已编码 6880/9000 条文本
已编码 7040/9000 条文本
已编码 7200/9000 条文本
已编码 7360/9000 条文本
已编码 7520/9000 条文本
已编码 7680/9000 条文本
已编码 7840/9000 条文本
已编码 8000/9000 条文本
已编码 8160/9000 条文本
已编码 8320/9000 条文本
已编码 8480/9000 条文本
已编码 86

#### upload your homework: use html file format
!jupyter nbconvert assignment-01-XXX-YYY.ipynb --to html --template classic --embed-images

In [1]:
!jupyter nbconvert assignment-01-25210980120-杨晨.ipynb --to html --template classic --embed-images

[NbConvertApp] Converting notebook assignment-01-25210980120-杨晨.ipynb to html
[NbConvertApp] Writing 430559 bytes to assignment-01-25210980120-杨晨.html
